# Multilingual NLP Language Models

Unigram and n-gram language models built from scratch across English and Japanese, applied to token prediction and manuscript recovery. This project explores three related language modeling tasks:

1. **English unigram model** — trained on Bram Stoker's *Dracula*, computing per-token probabilities from cleaned tokenized text.
2. **Japanese unigram model** — trained on E. Phillips Oppenheim's *The Great Impersonation* (Japanese edition), using the `janome` tokenizer to handle Japanese non-space-separated text and filtering to Japanese-script tokens only.
3. **English n-gram model** — bigram and fivegram models trained on Shakespeare, applied to a manuscript-recovery task predicting deleted tokens against a validation reference.

**Libraries:** `nltk`, `janome` (Japanese tokenization), `pickle` (model persistence).

**Skills demonstrated:** multilingual text preprocessing, unigram/n-gram frequency modeling, conditional probability estimation, and evaluation against ground truth.

---

## Part 1: English Unigram Language Model (Dracula)

**Objective:** Calculate the probability of each token in the corpus and rank the most probable words.

**Corpus:** Bram Stoker's *Dracula* (public domain via Project Gutenberg).

### Approach

Read the raw text, apply a data-cleaning pipeline (lowercase, strip punctuation, tokenize), compute unigram frequencies, and normalize to probabilities.

### Imports and text loading

In [1]:
# Import needed libraries
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download the ones needed later on
nltk.download('punkt')
nltk.download('stopwords')

# Function to load Dracula text, similar to my load_reviews pattern from assignment 1
def load_dracula_text(path):
    dracula_lines = []  # Start with empty list for the lines
    with open(path, encoding='utf-8') as file:  # Open file with UTF-8 encoding
        for line in file:
            stripped_line = line.strip()  # Remove leading/trailing whitespace
            if stripped_line:  # Only keep non-empty lines
                dracula_lines.append(stripped_line)
    return dracula_lines

# Use the function to load the text
dracula = load_dracula_text("pg345.txt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\<user>\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\<user>\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Data cleaning

Strip custom punctuation, lowercase, and tokenize with NLTK's `word_tokenize`.

In [2]:
# Create a data cleaning function
def cleanText(lines):
    custom_punct = ["“", "’", "--", "”"]
    punctuation = set(list(string.punctuation))
    stop_words = set(stopwords.words('english'))

    cleaned_lines = [] # Start with an empty list
    for line in lines:  # iterate over one line at a time
        tokens = word_tokenize(line) # tokenization occurs here
        cleaned = [
            word.lower() # make all words lowercase
            for word in tokens 
            if word.lower() not in stop_words  # drop any stop-words (after it converting to lowercase)
            and word not in punctuation # drop any punctuation
            and word not in custom_punct # drop those custom punctuations we want removed as well
        ]
        if cleaned:
            cleaned_lines.append(cleaned) # Appending the reviews with the changes we want (lowercase, punctuation removed, and stop-words removed)
    return cleaned_lines

# Run dracula through the cleanText function assigning back to dracula (instructions weren't clear if we should assign a new name)
dracula = cleanText(dracula)

### Probability calculation

Compute unigram frequency distribution and normalize to probabilities.

In [3]:
# References:
# https://stackoverflow.com/questions/54959340/nltk-language-modeling-confusion
# https://medium.com/mti-technology/n-gram-language-model-b7c2fc322799

# Import needed libraries
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

# Prepare training data for unigram model (n = 1)
n = 1
train_data, padded_vocab = padded_everygram_pipeline(n, dracula) # Use the desired names for train and padding

# Convert padded vocab to a list
padded_vocab = list(padded_vocab) # reassign back to padded_vocab

# Instantiate MLE unigram model
unigram_model = MLE(n)

# Fit the model with training data and padded_vocab
unigram_model.fit(train_data, padded_vocab)

# Calculate probabilities
unigram_probs = {word: unigram_model.score(word) for word in padded_vocab}

### Results: top 10 most probable tokens

In [4]:
# Print first 10 probabilities
for word, prob in list(unigram_probs.items())[:10]: #:10 includes all up to 10th one
    print(f"{word}: {prob}")

﻿the: 1.3585110718652357e-05
project: 0.0012090748539600599
gutenberg: 0.0004211384322782231
ebook: 0.00017660643934248063
dracula: 0.0005298193180274419
use: 0.0007335959788072273
anyone: 8.151066431191414e-05
anywhere: 0.00024453199293574245
united: 0.00024453199293574245
states: 0.00029887243581035183


---

## Part 2: Japanese Unigram Language Model

**Objective:** Same task, applied to Japanese text. The challenge here is that Japanese has no whitespace between words, so standard tokenization approaches don't work.

**Corpus:** E. Phillips Oppenheim, *The Great Impersonation* (Japanese translation, Project Gutenberg).

### Approach

Use the `janome` morphological analyzer to tokenize Japanese text, then filter to Japanese-script tokens (hiragana, katakana, kanji) — dropping any Latin characters, numerals, and punctuation that appear in the mixed source. Then compute unigram probabilities as in Part 1.

### Imports

In [1]:
# Import needed libraries
import re
from janome.tokenizer import Tokenizer
import nltk
nltk.download('stopwords')

# Function to load oppenheim text, same as load text function from assignment 2A just opening this new file
def load_oppenheim_text(path):
    oppenheim_lines = [] # Start with empty list for the lines
    with open(path, encoding='utf-8') as file: # Open file with UTF-8 encoding
        for line in file:
            stripped_line = line.strip() # Remove leading/trailing whitespace
            if stripped_line:  # Only keep non-empty lines
                oppenheim_lines.append(stripped_line)
    return oppenheim_lines

# Use the function to load the text
oppenheim = load_oppenheim_text("pg34158.txt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\<user>\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Japanese-script detection helper

In [2]:
import re
def is_japanese(word):
    """
    Determine if a word is written in Japanese script.

    This function checks if the given word contains any Japanese characters (Hiragana, Katakana, or Kanji)
    and does not contain any Latin alphabet characters. If the word contains Japanese characters and no
    Latin characters, it is considered a Japanese word.

    Parameters:
    word (str): The word to check.

    Returns:
    bool: True if the word is Japanese, False otherwise.
    """
    # Regex to identify if the word contains any Japanese character
    # Hiragana: U+3040-U+309F, Katakana: U+30A0-U+30FF, Kanji: U+4E00-U+9FAF
    jap_regex = r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]'

    # Regex to identify if the word contains any Latin alphabet character
    eng_regex = r'[a-zA-Z]'

    # If the word contains any Japanese characters and no Latin characters, it's considered Japanese
    if re.search(jap_regex, word) and not re.search(eng_regex, word):
        return True
    else:
        return False

### Data cleaning with janome tokenizer

`janome.Tokenizer` performs morphological analysis, splitting Japanese sentences into their constituent morphemes without whitespace boundaries.

In [3]:
# Clean and tokenize Japanese text
def cleanText(lines):
    tokenizer = Tokenizer() # Create a tokenizer instance
    stop_words = set(['の', 'に', 'は', 'を', 'た', 'が', 'で', 'て', 'と', 'し', 'れ', 'さ', 'ある', 'いる', 'も', 'する',
                      'から', 'な', 'こと', 'として', 'い', 'や', 'する', 'など', 'なり', 'なく', 'まで', 'だ', 'へ',
                      'か', 'だっ', 'その', 'あっ', 'よう', 'また', 'もの', 'という', 'あり', 'まし', 'ませ', 'う',
                      'ない', 'ながら', 'なけれ', 'なし', 'ず', 'なっ', 'れる', 'られ', 'なる', 'べき', 'ほど', 'ます',
                      'てる', 'なら', 'せる', 'され', 'して']) # Set the stop words we specifically want as given in instructions
    punctuation = set(['。', '、', '？', '！', '「', '」', '『', '』', '（', '）', '；', '：', '-']) # Set the punctuation we specifically want as given in instructions

    cleaned_lines = [] # Start with an empty list
    for line in lines:
        spaced_line = ''.join([f"{char} " if is_japanese(char) else char for char in line]).strip() # try this because the spaces I currently have does not work as needed
        tokens = [token.surface for token in tokenizer.tokenize(spaced_line)] # tokenization occurs here - given in instructions
        cleaned = [
            char # split each token into characters
            for token in tokens
            for char in token  # break into characters
            if is_japanese(char) and char not in stop_words and char not in punctuation
        ] # drop punctuation and stop words we defined, and those in the is_japanese function above
        cleaned_lines.append(cleaned) # Appending back with the changes removed
    return cleaned_lines

# Run oppenheim through the cleanText function assigning back to oppenheim (instructions weren't clear if we should assign a new name)
oppenheim = cleanText(oppenheim)

### Probability calculation

In [4]:
# Used same code from Assignment 2A just changed dracula to oppenheim
# References:
# https://stackoverflow.com/questions/54959340/nltk-language-modeling-confusion
# https://medium.com/mti-technology/n-gram-language-model-b7c2fc322799

# Import needed libraries
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

# Prepare training data for unigram model (n = 1)
n = 1
train_data, padded_vocab = padded_everygram_pipeline(n, oppenheim) # Use the desired names for train and padding

# Convert padded vocab to a list
padded_vocab = list(padded_vocab) # reassign back to padded_vocab

# Instantiate MLE unigram model
unigram_model = MLE(n)

# Fit the model with training data and padded_vocab
unigram_model.fit(train_data, padded_vocab)

# Calculate probabilities
unigram_probs = {word: unigram_model.score(word) for word in padded_vocab}

### Results: top 10 most probable Japanese tokens

In [5]:
# Print top 10 probabilities - similar format to assignment 2a
for word, prob in sorted(unigram_probs.items(), key=lambda x: x[1], reverse = True)[:10]:  # Top 10 sorted
    print(f"{word}: {prob}")

っ: 0.03913621814856383
る: 0.02861952861952862
ら: 0.025501932909340316
こ: 0.024036662925551816
ー: 0.019131645674855553
ま: 0.018404206675811614
あ: 0.017915783347882113
す: 0.017915783347882113
り: 0.017271480234443196
わ: 0.017177952363137548


---

## Part 3: N-gram Language Models for Manuscript Recovery

**Objective:** Train bigram and fivegram language models on Shakespeare's corpus, then use them to predict tokens that have been replaced with a `<DELETED>` placeholder in a corrupted test manuscript. Evaluate the accuracy of the reconstruction against a known validation reference.

### Approach

1. Build a text-loading and cleaning pipeline reusable across corpora.
2. Generate n-gram sequences with NLTK.
3. Compute conditional frequency and probability distributions.
4. Predict the most probable next word given a context window.
5. Scan the test manuscript for `<DELETED>` markers, replace each with the model's prediction based on the preceding tokens, and score against validation.

### 3.1 — Imports

In [1]:
# Import needed libraries
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams

### 3.2 — Text loading

In [2]:
# Function to load the text, similar to my load_reviews pattern from assignment 1/2a except instead of reading each line it's into one string
def load_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:  # Open file with UTF-8 encoding
        return file.read()  # Return the entire content as a single string

# Use the function to load the text
train_text = load_text("WS_train.txt")

### 3.3 — Text cleaning

In [3]:
# Create the cleaning function - made similar to the previous assignments cleaning functions but we need a specific order of things
def clean_text(text):
    stop_words = set(stopwords.words('english')) # Setting the list stopwords given in the instructions and giving it a name to call in the function
    punctuation = set(list(string.punctuation)) # Setting the list punctuation given in the instructions and giving it a name to call in the function

    text = text.replace("<DELETED>", "DELETED_PLACEHOLDER")
    tokens = word_tokenize(text) # tokensize only for right now
    cleaned = [] # Empty list
    for word in tokens:
        if word == 'DELETED_PLACEHOLDER':
            cleaned.append("<DELETED>")
        else:
            word_low = word.lower() # now lowercase
            if word_low not in stop_words and word_low not in punctuation:
                cleaned.append(word_low)
    
    return cleaned

# Run the clean function to get the cleaned train text
cleaned_train_text = clean_text(train_text)

### 3.4 — N-gram generation

In [4]:
# References:
# https://www.askpython.com/python/examples/n-grams-python-nltk
# https://www.delftstack.com/howto/python/ngrams-python/#google_vignette
# https://thinkingneuron.com/how-to-generate-n-grams-in-python/

# Create the function that will return n-grams
def create_ngrams(tokens, n):
    start_tokens = ['<s>'] * (n - 1) # start of the sentence and n-1 special token for the start
    end_tokens = ['</s>'] # end of the sentence
    padded_tokens = start_tokens + tokens + end_tokens # pad <s>'s depending on the n-gram so 3-1 = 2 so 2 <s>'s will be padded for a trigram
    return list(ngrams(padded_tokens, n)) # n needed for order of n-grams (so we can create different # of grams)

# Create bigrams with the create_ngrams function
train_bigrams = create_ngrams(cleaned_train_text, 2)

# Create fivegrams with the create_ngrams function
train_fivegrams = create_ngrams(cleaned_train_text, 5)

### 3.5 — Vocabulary construction

Build the vocabulary from training text, excluding the `<DELETED>` placeholder token as a safeguard so the model never predicts it.

In [5]:
# Create a function excluding <DELETED> as a safeguard for test text
def build_vocab(tokens):
    return set(token for token in tokens if token != '<DELETED>')

# Create vocab which makes sure <DELETED> is not in vocabulary
vocab = build_vocab(cleaned_train_text)

### 3.6 — Frequency distribution

In [6]:
# Import needed libraries
from nltk import FreqDist, ConditionalFreqDist, ConditionalProbDist, MLEProbDist

In [7]:
# Reference: https://www.nltk.org/api/nltk.probability.FreqDist.html
# Create a function that will calculate the frequency of n-grams
def calculate_ngram_freq(ngrams_list):
    return FreqDist(ngrams_list) # returns the frequency distribution of a n-gram

# Create a bigram frequency distribution
bigram_freq_dist = calculate_ngram_freq(train_bigrams)

# Create a fivegram frequency distribution
fivegram_freq_dist = calculate_ngram_freq(train_fivegrams)

### 3.7 — Conditional probability estimation

In [8]:
# References:
# https://tedboy.github.io/nlps/generated/generated/nltk.ConditionalProbDist.html
# https://www.nltk.org/api/nltk.probability.MLEProbDist.html?highlight=probability+probdist#nltk.probability.MLEProbDist
# https://www.geeksforgeeks.org/python-nltk-nltk-tokenize-conditionalfreqdist/
# https://stackoverflow.com/questions/32676319/new-to-nltk-having-trouble-with-conditional-frequency

# Create a function that will estimate conditional probability and probability distribution for a n-gram
def estimate_ngram_probabilities(ngrams_list):
    cond_freq = ConditionalFreqDist() # will map each (n-1)-gram to a frequency distribution of next word possibilites
    for ngram in ngrams_list:
        context = ngram[:-1]  # extract tthe first n-1 of words of the ngram
        target = ngram[-1]    # extract the last word in n-1 ngram
        cond_freq[context][target] += 1  # increment the count of the target word given the ngram wanted
    return ConditionalProbDist(cond_freq, MLEProbDist) # turn raw frequencies from cond_freq into actual probabilities using MLE

# Create bigram probability distribution
bigram_prob_dist = estimate_ngram_probabilities(train_bigrams)

# Create fivegram probability distribution
fivegram_prob_dist = estimate_ngram_probabilities(train_fivegrams)

### 3.8 — Next-word prediction

In [9]:
# Import needed library
from nltk.probability import ConditionalProbDist

In [10]:
# Reference:  https://stackoverflow.com/questions/16310015/what-does-this-mean-key-lambda-x-x1
# Create a function that predicts the most probable next word(s) using ConditionalProbDist
def predict_next_word(context, cpd, top_n=1):
    if context not in cpd:
        return "<UNK>" if top_n == 1 else [("<UNK>", 0.0)]  # handle unknown contexts so we return UNK

    # Get the distribution for the given context
    predict_prob_dist = cpd[context]
    
    # If top_n == 1, the function will then return a single string (most probable word)
    # Otherwise, returns a list of tuples
    if top_n == 1:
        # Return the word with the highest probability
        return predict_prob_dist.max()
    else:
        # Return the top_n most probable words with their probabilities, sorted in descending order of top_n
        return sorted([(prob_word, predict_prob_dist.prob(prob_word)) for prob_word in predict_prob_dist.samples()],
                      key=lambda x: x[1], reverse=True)[:top_n]        

### 3.9 — Text correction function

Scan the corrupted text for `<DELETED>` markers and replace each with the model's most probable next-word prediction given the preceding context.

In [11]:
# Create a function that searches for DELETED placeholders uses the predict_next_word function we created above to find the best probable replacement
def correct_text_with_ngrams(text_data, cpd, n):
    corrected_text = []  # list of corrected tokens where DELETED placeholders have been replaced, empty for right now
    context_window = n - 1  # number of words in context for prediction (utilizing the n-1 we defined earlier in above functions)

    for i in range(len(text_data)):
        if text_data[i] == "<DELETED>":
            # Gather the context - last number n-1 tokens before the current position
            context_start = max(0, i - context_window)
            context = tuple(corrected_text[context_start:i])

            # Pad the context with <s> if it's too short (similar to what we did in a previous function with padding s)
            if len(context) < context_window:
                context = ('<s>',) * (context_window - len(context)) + context

            # Predict the most probable replacement word
            prediction = predict_next_word(context, cpd, top_n=1) # using context and top_n=1 from the predict_next_word function
            corrected_text.append(prediction)
        else:
            corrected_text.append(text_data[i])

    return corrected_text

### 3.10 — Load and prepare the test manuscript

In [12]:
# Load the test text
test_text = load_text("WS_test.txt")

# Clean the test text
cleaned_test_text = clean_text(test_text)  # retains <DELETED> placeholders
deleted_count = cleaned_test_text.count("<DELETED>")
print(f"<DELETED> tokens in test set: {deleted_count}")

<DELETED> tokens in test set: 1740


### 3.11 — Apply the bigram model

In [13]:
# Apply correction using bigram model (correct_text_with_ngrams function)
corrected_test_text_bigram = correct_text_with_ngrams(cleaned_test_text, bigram_prob_dist, 2) # making sure to put the correct n for the ngram

### 3.12 — Apply the fivegram model

In [14]:
# Apply correction using fivegram model (correct_text_with_ngrams function)
corrected_test_text_fivegram = correct_text_with_ngrams(cleaned_test_text, fivegram_prob_dist, 5) # making sure to put the correct n for the ngram

### 3.13 — Evaluation against validation reference

In [15]:
def calculate_accuracy(test_tokens, corrected_tokens, validation_tokens):
    if (len(test_tokens) != len(corrected_tokens)) or (len(test_tokens) != len(validation_tokens)):
        print("Test Tokens, Validation Token and Corrected Tokens must have the same length")
        return

    correct_predictions = 0
    all_predictions = 0
    for i in range(len(test_tokens)):
        if test_tokens[i] == '<DELETED>':
            all_predictions+=1
            if corrected_tokens[i] == validation_tokens[i]:
                   correct_predictions+=1

    return correct_predictions/all_predictions

In [16]:
# Load the validation text using the load_text function
validation_text = load_text("WS_validation.txt")

# Clean the validation text using the clean_text function
cleaned_validation_text = clean_text(validation_text)

# Evaluate accuracy for bigram model - using the calculate_accuracy function given above
bigram_accuracy = calculate_accuracy(
    test_tokens = cleaned_test_text,
    corrected_tokens = corrected_test_text_bigram,
    validation_tokens = cleaned_validation_text
) # Use the cleaned and corrected data we made earlier in the file, each token (corrected_tokens, test_tokens, etc) dictate which text or gram that should be used

# Evaluate accuracy for fivegram model
fivegram_accuracy = calculate_accuracy(
    test_tokens = cleaned_test_text,
    corrected_tokens = corrected_test_text_fivegram,
    validation_tokens = cleaned_validation_text
) # Use the cleaned and corrected data we made earlier in the file, each token (corrected_tokens, test_tokens, etc) dictate which text or gram that should be used

---

## Summary

This project demonstrates:

- **Language-agnostic pipeline design** — the same conceptual workflow (load → clean → tokenize → estimate probabilities) applied across English and Japanese with language-appropriate tokenization.
- **Multilingual tokenization** — using `janome` for Japanese morphological analysis where whitespace-based tokenization fails.
- **N-gram probability estimation** — extending unigram models to bigram and fivegram conditional distributions.
- **Applied use case** — manuscript recovery, evaluating predictions against a validation reference to measure correctness.